# Gym Booking Automation

* Log in automatically

* Book specific gym classes

* Handle waitlists

* Deal with network errors like a pro

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

In [2]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [3]:
#stay open using the Chrome option:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option("detach", True)
#Create a folder for the Chrome Profile selenium AND will use it every time
user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
#use the directory specified to store a "profile".
chrome_options.add_argument(f"--user-data-dir={user_data_dir}")

driver = webdriver.Chrome(options=chrome_options)
driver.get(GYM_URL)

In [4]:
#this will retries instead of immediately raising an exception
wait = WebDriverWait(
    driver,
    15, #wait 15 seconds
    poll_frequency=0.2, #wait 0.2 seconds
    ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
)

# Wait until the language button is clickable
wait.until(
    #we get the element that we are looking
    EC.element_to_be_clickable((By.ID, 'login-button'))
#and then make click    
).click()

In [5]:
#now fill the inputs with our data
email_input = driver.find_element(By.ID, value='email-input')
email_input.send_keys(ACCOUNT)
password_input = driver.find_element(By.ID, value='password-input')
password_input.send_keys(PASSWORD)

#AND SEND IT
driver.find_element(By.ID, value='submit-button').click()

# Now once we access the instructions says

**Book Your First Class - Tuesday at 6pm**

In [ ]:
#find the buttons
#Lets see the condition on the value
# first we are looking for all buttons then []
# means the ID starts with book-button-.

list_buttons = driver.find_elements(By.CSS_SELECTOR, value="button[id^='book-button-']")
for i in range(0,5):
    print("ID:", list_buttons[i].get_attribute("id"))
    print("Text:", list_buttons[i].text.strip())
    print("Tag:", list_buttons[i].tag_name)
    print("Aria busy:", list_buttons[i].get_attribute("aria-busy"))
    print("Aria disabled:", list_buttons[i].get_attribute("aria-disabled"))
    print("Enabled:", list_buttons[i].is_enabled())
    print("--------------------")

# The problem (for specific classes) is that we have a lot of buttons 
# we can see all of them until we reach the date we wanted, but
# we do not know what class will be (it could be any and we do not know that) 

ID: book-button-yoga-2026-11-06-1700
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-spin-2026-11-06-1800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-hiit-2026-11-06-1900
Text: Join Waitlist
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-yoga-2026-11-07-0700
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------
ID: book-button-spin-2026-11-07-0800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------


### (Just for the future this method of getting all the buttons will help us to book more clasess, but we need to figure if they are enable to click)

In [ ]:
for button in list_buttons:
    #we join all the text and ignore uppercase or lowercase
    button_text = " ".join(button.text.split()).casefold()
    #and compare first if it says book class, and can be click it
    if button_text == "book class" and button.is_enabled():
        #then we click it to reserve it
        #(for now we just print the id of the ones that are availables)
        print("Available:", button.get_attribute("id"))

Available: book-button-yoga-2026-11-06-1700
Available: book-button-spin-2026-11-06-1800
Available: book-button-yoga-2026-11-07-0700
Available: book-button-spin-2026-11-07-0800
Available: book-button-yoga-2026-11-07-1700
Available: book-button-spin-2026-11-07-1800
Available: book-button-hiit-2026-11-07-1900
Available: book-button-yoga-2026-11-08-0700
Available: book-button-hiit-2026-11-08-0900
Available: book-button-yoga-2026-11-08-1700
Available: book-button-spin-2026-11-08-1800
Available: book-button-hiit-2026-11-08-1900
Available: book-button-yoga-2026-11-09-0700
Available: book-button-hiit-2026-11-09-0900
Available: book-button-yoga-2026-11-09-1700
Available: book-button-hiit-2026-11-09-1900
Available: book-button-yoga-2026-11-10-0700
Available: book-button-spin-2026-11-10-0800
Available: book-button-hiit-2026-11-10-0900
Available: book-button-yoga-2026-11-10-1700
Available: book-button-spin-2026-11-10-1800
Available: book-button-hiit-2026-11-10-1900
Available: book-button-yoga-2026

# For Tuesday (specific class) First we need to get the date

In [38]:
from datetime import date, timedelta, datetime

In [8]:
#calculate next tuesday
today = date.today()

# tuesday = 1 (Monday is 0)
days_until_tuesday = (1 - today.weekday()) % 7

# If today is tuesday, select next tuesday
if days_until_tuesday == 0:
    days_until_tuesday = 7

#now we have the days, we need the date, so we are going to add the days
#for the right one
next_date = today + timedelta(days=days_until_tuesday)
#and we convert the date into a string
date_string = next_date.strftime("%Y-%m-%d")

print(date_string)

2026-11-10


# Then updated the search by only one element with that id

In [25]:
# Now we updated the condition
# For the second part means the ID contains the selected date. (plus the hour that in this case we can put it)
button = driver.find_element(By.CSS_SELECTOR, 
                             value=f"button[id^='book-button-'][id*='{date_string}-1800']")

print("ID:", button.get_attribute("id"))
print("Text:", button.text.strip())
print("Tag:", button.tag_name)
print("Aria busy:", button.get_attribute("aria-busy"))
print("Aria disabled:", button.get_attribute("aria-disabled"))
print("Enabled:", button.is_enabled())
print("--------------------")

ID: book-button-spin-2026-11-10-1800
Text: Book Class
Tag: button
Aria busy: false
Aria disabled: None
Enabled: True
--------------------


In [26]:
# we do click
button.click()

In [29]:
#now the result
button = driver.find_element(By.CSS_SELECTOR, 
                             value=f"button[id^='book-button-'][id*='{date_string}-1800']")
#as we see it is the class booked and the button is disable 
# that means we can not do click again
print("ID:", button.get_attribute("id"))
print("Text:", button.text.strip())
print("Tag:", button.tag_name)
print("Aria busy:", button.get_attribute("aria-busy"))
print("Aria disabled:", button.get_attribute("aria-disabled"))
print("Enabled:", button.is_enabled())
print("--------------------")

ID: book-button-spin-2026-11-10-1800
Text: Booked
Tag: button
Aria busy: None
Aria disabled: None
Enabled: False
--------------------


# As the instruction says we need now to **Check if a class is already booked** but that already was done, now we only need to updated our condition

Instaed of printing the ones that are available we are going to print

* Check if a class is already booked (button reads "Booked")
  * Example:
    * Already booked: Spin Class on Tue, Aug 12
* Check if you're on the waitlist (button reads "Waitlisted")
  * Example:
    * Already on waitlist: HIIT Class on Tue, Aug 12
* Join the waitlist if the class is full (button says "Join Waitlist")
  * Example:
    * Joined waitlist for: Yoga Class on Tue, Aug 12

In [ ]:
# all the data is on
print(button.get_attribute("id"))
#book-button-yoga-2026-11-06-1700

#with that we can split it to get the class, year, month and day
all_data_button = button.get_attribute("id").split('-')
print(all_data_button)
#now the data
schedule_class = all_data_button[2]
print(schedule_class)

schedule_year = int(all_data_button[3])
print(schedule_year)

schedule_day = int(all_data_button[5])
print(schedule_day)

#what about the month, we wanted text not number
schedule_month = int(all_data_button[4])
#we need to pass like this (year, month, day).(%B) name of month
#it is a lot (and the data need to be on integer)
print(datetime(schedule_year, schedule_month, schedule_day).strftime("%B"))

book-button-spin-2026-11-10-1800
['book', 'button', 'spin', '2026', '11', '10', '1800']
spin
2026
10
November


#Well for fortune we can look for parent
'''
Child =  card element

Parent = the element directly wrapping the card (e.g., a div)

Grandparent = the wrapper around the parent one level up


Our structure is like this

< div id="day-group-today-(fri,-nov-6)">

    < h2 id="day-title-today-(fri,-nov-6)">

        Today (Fri, Nov 6)

    </ h2>

    <!-- Other elements and divs -->

    < button id="book-button-yoga-2026-11-06-1700">

        Book class

    </ button>

</ div>

So lets se how we do it
'''

In [47]:
day_group = button.find_element(
    By.XPATH,
    "ancestor::div[starts-with(@id, 'day-group-')][1]"
)
# ancestor::div searches the parent elements and their parents.
# starts-with(@id, 'day-group-') finds the div whose ID starts with day-group-.
# [1] selects the closest matching ancestor.

In [ ]:
#witht that now we can find the heading
day_title = day_group.find_element(
    By.CSS_SELECTOR, "h2"
).text.strip()
#and that is the date (for each button we can have it like that)
print(day_title)

Tue, Nov 10


# We can keep how we get the class name, the date is going to be like this so lets do it for all the buttons but we are going to do it on a function

In [1]:
#libraries
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from datetime import date, timedelta 

In [2]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [3]:
def chrome_configuration():
    #stay open using the Chrome option:
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_experimental_option("detach", True)
    #Create a folder for the Chrome Profile selenium AND will use it every time
    user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
    #use the directory specified to store a "profile".
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")
    return chrome_options

In [4]:
def make_connection(chrome_options):
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(GYM_URL)
    return driver

In [5]:
def log_in(wait):  
    # Wait until the language button is clickable
    wait.until(
        #we get the element that we are looking
        EC.element_to_be_clickable((By.ID, 'login-button'))
    #and then make click    
    ).click()
    
    #second wait
    # Wait until the email input is visible
    email_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'email-input'))
    )
    email_input.send_keys(ACCOUNT)

    # Wait until the password input is visible
    password_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'password-input'))
    )
    password_input.send_keys(PASSWORD)

    # Wait until the submit button is clickable
    wait.until(
        EC.element_to_be_clickable((By.ID, 'submit-button'))
    ).click()

In [6]:
def find_all_buttons(wait):
    #we will wait until the elements (all the buttons) are displayed (loaded)
    list_buttons = wait.until(
        EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, 
        "button[id^='book-button-']"
        ))
    )
    return list_buttons

In [7]:
def get_date_title(button):
    # Get date from the ancestor
    day_group = button.find_element(
        By.XPATH,
        "ancestor::div[starts-with(@id, 'day-group-')][1]"
    )

    day_title = day_group.find_element(
        By.CSS_SELECTOR, "h2"
    ).text.strip()
    
    return day_title

In [8]:
def check_classes(all_buttons, class_names):
    for button in all_buttons:
        button_text = " ".join(button.text.split()).casefold()
        #compare first if the class is other case except for book it
        if button_text != "Book Class":
            #then we get the date and class name
            date_label = get_date_title(button)
            schedule_class = button.get_attribute("id").split('-')[2]
            #if the class name is not on the dictionary, then we put the class on title case
            class_name = class_names.get(schedule_class, schedule_class.title())
            #now we just compare
            if button_text == "booked":
                print(f"Already booked: {class_name} Class on {date_label}")
            elif button_text == "waitlisted":
                print(f"Already on waitlist: {class_name} Class on {date_label}")
            #Join the waitlist (first we do click and the we can show the data)
            elif button_text == "join waitlist":
                button.click()
                print(f"Clicked Join Waitlist for: {class_name} Class on {date_label}")
            

In [9]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait)

    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printnig the results for the classes')
    #check the classes that are not available
    check_classes(all_buttons, class_names)

In [10]:
main()

Checking the classes...
Printnig the results for the classes
Clicked Join Waitlist for: HIIT Class on Today (Fri, Nov 6)
Clicked Join Waitlist for: HIIT Class on Tomorrow (Sat, Nov 7)
Clicked Join Waitlist for: Spin Class on Sun, Nov 8
Clicked Join Waitlist for: Spin Class on Mon, Nov 9
Clicked Join Waitlist for: Spin Class on Mon, Nov 9
Already booked: Spin Class on Tue, Nov 10
Clicked Join Waitlist for: Spin Class on Thu, Nov 12


In [12]:
main()

Checking the classes...
Printnig the results for the classes
Already on waitlist: HIIT Class on Today (Fri, Nov 6)
Already on waitlist: HIIT Class on Tomorrow (Sat, Nov 7)
Already on waitlist: Spin Class on Sun, Nov 8
Already on waitlist: Spin Class on Mon, Nov 9
Already on waitlist: Spin Class on Mon, Nov 9
Already booked: Spin Class on Tue, Nov 10
Already on waitlist: Spin Class on Thu, Nov 12


# It works, we add some waiters for the objects to load, now after printing and checking the classes that we are, now we will print this:
--- BOOKING SUMMARY ---

Classes booked: 0

Waitlists joined: 0

Already booked/waitlisted: 1

Total Tuesday 6pm classes processed: 1 


In [22]:
#add a neew class to make dictionary of data
def addclasses(class_name, date_label):
    dict_classes = {
        'class': class_name,
        'date': date_label,
    }
    return dict_classes

In [ ]:
# we will updated this class
def check_classes(all_buttons, class_names):
    #counters
    counter_booked = 0
    counter_joined_listed = 0
    counter_booked_waitlisted = 0
    #list of the classes (list of dictionary)
    list_processed_classes = []
    

    for button in all_buttons:
        button_text = " ".join(button.text.split()).casefold()

        date_label = get_date_title(button)
        schedule_class = button.get_attribute("id").split('-')[2]
        class_name = class_names.get(schedule_class, schedule_class.title())

        if button_text == "Book Class":
            #this is for the class be able to book
            #but for now we will not do click but is just like this
            '''
            button.click()
            print(f"Clicked Booked Class for: {class_name} Class on {date_label}")
            '''
            counter_booked+=1

        elif button_text == "booked":
            counter_booked_waitlisted+=1
            print(f"Already booked: {class_name} Class on {date_label}")
            list_processed_classes.append(addclasses(class_name, date_label))

        elif button_text == "waitlisted":
            counter_booked_waitlisted+=1
            print(f"Already on waitlist: {class_name} Class on {date_label}")

            list_processed_classes.append(addclasses(class_name, date_label))
        #Join the waitlist (first we do click and the we can show the data)
        elif button_text == "join waitlist":
            button.click()
            counter_joined_listed+=1
            print(f"Clicked Join Waitlist for: {class_name} Class on {date_label}")
            list_processed_classes.append(addclasses(class_name, date_label))
    return list_processed_classes, counter_booked, counter_booked_waitlisted, counter_joined_listed  

In [32]:
def summary(list_processed_classes, counter_booked, counter_booked_waitlisted, counter_joined_listed ):
    print('\n--- BOOKING SUMMARY ---')
    print(f"Classes booked:{counter_booked}")
    print(f"Waitlists joined:{counter_joined_listed}") 
    print(f"Already booked/waitlisted:{counter_booked_waitlisted}")
    #for the classes we print them:
    print('Total')
    for classes in list_processed_classes:
        print(f'Class: {classes["class"]} {classes["date"]}')
    print(f'Processed: {len(list_processed_classes)}')

In [33]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait)

    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printnig the results for the classes')
    #check the classes that are not available
    list_processed_classes, counter_booked, counter_booked_waitlisted, counter_joined_listed  = check_classes(all_buttons, class_names)

    summary(list_processed_classes, counter_booked, counter_booked_waitlisted, counter_joined_listed )

In [34]:
main()

Checking the classes...
Printnig the results for the classes
Already on waitlist: HIIT Class on Today (Fri, Nov 6)
Already on waitlist: HIIT Class on Tomorrow (Sat, Nov 7)
Already on waitlist: Spin Class on Sun, Nov 8
Already on waitlist: Spin Class on Mon, Nov 9
Already on waitlist: Spin Class on Mon, Nov 9
Already booked: Spin Class on Tue, Nov 10
Already on waitlist: Spin Class on Thu, Nov 12

--- BOOKING SUMMARY ---
Classes booked:0
Waitlists joined:0
Already booked/waitlisted:7
Total
Class: HIIT Today (Fri, Nov 6)
Class: HIIT Tomorrow (Sat, Nov 7)
Class: Spin Sun, Nov 8
Class: Spin Mon, Nov 9
Class: Spin Mon, Nov 9
Class: Spin Tue, Nov 10
Class: Spin Thu, Nov 12
Processed: 7


# Now we need to updated again our code to do the next:
* Book EVERY Tuesday AND Thursday 6pm class
* Keep track of all classes processed
* Print a detailed list of what happened

In [36]:
from datetime import date, timedelta

In [39]:
def check_resting_days(amount,today):
    days_until = (amount - today.weekday()) % 7
    if days_until == 0:
        days_until = 7
    return days_until

In [41]:
#calculate for tuesday and thursday
today = date.today()

days_until_tuesday = check_resting_days(1, today)
days_until_thursday = check_resting_days(3, today)

In [42]:
print(days_until_tuesday, days_until_thursday)
print(type(days_until_tuesday),type(days_until_thursday))

4 6
<class 'int'> <class 'int'>


In [43]:
def convert_date(days_until, today):
    next_date = today + timedelta(days=days_until)
    #and we convert the date into a string
    date_string = next_date.strftime("%Y-%m-%d")
    return date_string

In [57]:
next_tuesday = convert_date(days_until_tuesday, today)
next_thursday = convert_date(days_until_thursday, today)

In [45]:
print(next_tuesday, next_thursday)
print(type(next_tuesday),type(next_thursday))

2026-11-10 2026-11-12
<class 'str'> <class 'str'>


In [60]:
#we can compare the id for the last elements
#if we split the id we can check
#button.get_attribute("id").split('-')
string_temp = 'book-button-spin-2026-11-12-1800'
id_splited = string_temp.split('-')
print(id_splited)
id_splited = string_temp.split('-')[3:]
print(id_splited)
id_joined = ('-').join(id_splited)
print(id_joined)
#all at once
id_joined= ('-').join((string_temp.split('-')[3:]))
print(id_joined)

['book', 'button', 'spin', '2026', '11', '12', '1800']
['2026', '11', '12', '1800']
2026-11-12-1800
2026-11-12-1800


In [58]:
#example
next_thursday+='-1800'
print(next_thursday)

2026-11-12-1800


In [ ]:
#and then we compare
if id_joined == next_thursday:
    print('Class found')
    #print("Text:", button.text.strip())

Class found


# Ok with that we can start

In [4]:
#libraries
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from datetime import date, timedelta 

In [5]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [6]:
def chrome_configuration():
    #stay open using the Chrome option:
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_experimental_option("detach", True)
    #Create a folder for the Chrome Profile selenium AND will use it every time
    user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
    #use the directory specified to store a "profile".
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")
    return chrome_options

In [7]:
def make_connection(chrome_options):
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(GYM_URL)
    return driver

In [8]:
def log_in(wait):  
    # Wait until the language button is clickable
    wait.until(
        #we get the element that we are looking
        EC.element_to_be_clickable((By.ID, 'login-button'))
    #and then make click    
    ).click()
    
    #second wait
    # Wait until the email input is visible
    email_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'email-input'))
    )
    email_input.send_keys(ACCOUNT)

    # Wait until the password input is visible
    password_input = wait.until(
        EC.visibility_of_element_located((By.ID, 'password-input'))
    )
    password_input.send_keys(PASSWORD)

    # Wait until the submit button is clickable
    wait.until(
        EC.element_to_be_clickable((By.ID, 'submit-button'))
    ).click()

In [9]:
def find_all_buttons(wait):
    #we will wait until the elements (all the buttons) are displayed (loaded)
    list_buttons = wait.until(
        EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, 
        "button[id^='book-button-']"
        ))
    )
    return list_buttons

In [10]:
def get_date_title(button):
    # Get date from the ancestor
    day_group = button.find_element(
        By.XPATH,
        "ancestor::div[starts-with(@id, 'day-group-')][1]"
    )

    day_title = day_group.find_element(
        By.CSS_SELECTOR, "h2"
    ).text.strip()
    
    return day_title

In [76]:
def check_resting_days(amount,today):
    days_until = (amount - today.weekday()) % 7
    '''
    if days_until == 0:
        days_until = 7
    '''
    return days_until

In [12]:
def convert_date(days_until, today):
    next_date = today + timedelta(days=days_until)
    #and we convert the date into a string
    date_string = next_date.strftime("%Y-%m-%d")
    return date_string

In [75]:
def check_day(button, next_tuesday, next_thursday):
    #all at once
    date_tag = ('-').join((button.split('-')[3:]))
    #and then we compare
    #We can return the Boolean expression directly. There's no need for an additional if.
    return date_tag in (next_tuesday, next_thursday)

In [14]:
#so ok we are refactorizing again
def check_classes(driver, all_buttons, class_names, next_tuesday, next_thursday):
    #we are going to make a list of dictionary
    #this will delete all the other variables
    processed_classes = []
    #and a counter
    total_target_classes = 0

    # Save button IDs before interacting with the page
    #because if we change some button (we can have an StaleElementReferenceException, since we change the obj)
    button_ids = [
        button.get_attribute("id")
        for button in all_buttons
    ]
    
    for button_id in button_ids:

        # Find the element again to avoid stale references
        button = driver.find_element(By.ID, button_id)
        #all again
        button_text = " ".join(button.text.split()).casefold()
        date_label = get_date_title(button)
        schedule_class = button_id.split("-")[2]
        class_name = class_names.get(schedule_class, schedule_class.title())
        #new variable for the dictionary
        status = None

        #new booking
        if button_text == "book class":
            #check if is the day tuesday or thursday
            if check_day(button_id, next_tuesday, next_thursday):
                button.click()
                print(f"Clicked Booked Class for: {class_name} Class on {date_label}")
                status = "New Booking"
                #counter of all the classes
                total_target_classes+=1
            
        elif button_text == "join waitlist":
            #check if is the day tuesday or thursday
            if check_day(button_id, next_tuesday, next_thursday):                    
                button.click()
                print(f"Joined waitlist for: {class_name} Class on {date_label}")
                status = "New Waitlist"
                #counter of all the classes
                total_target_classes+=1

        #check the rest   
        elif button_text == "booked":
            print(f"Already booked: {class_name} Class on {date_label}")
            status = "Already Booked"

        elif button_text == "waitlisted":
            print(f"Already on waitlist: {class_name} Class on {date_label}")
            status = "Already Waitlisted"

        #check for status
        if status is not None:
            processed_classes.append({
                "class": class_name,
                "date": date_label,
                "status": status
            })
        
    return processed_classes, total_target_classes 

In [ ]:
def summary(processed_classes, total_target_classes):

    # Calculate counters
    new_bookings = sum(
        c["status"] == "New Booking"
        for c in processed_classes
    )

    new_waitlists = sum(
        c["status"] == "New Waitlist"
        for c in processed_classes
    )

    already_processed = sum(
        c["status"] in ("Already Booked", "Already Waitlisted")
        for c in processed_classes
    )

    #print
    print('\n--- BOOKING SUMMARY ---')
    print(f"Classes booked:{new_bookings}")
    print(f"Waitlists joined:{new_waitlists}") 
    print(f"Already booked/waitlisted:{already_processed}")
    #for the classes we print them:
    print(
        f"Total Tuesday & Thursday 6pm classes: {total_target_classes}"
    )

    print("\n--- DETAILED CLASS LIST ---")

    for class_info in processed_classes:

        print(
            f'  • [{class_info["status"]}] '
            f'{class_info["class"]} Class on {class_info["date"]}'
        )
    return already_processed


In [ ]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #calculate for tuesday and thursday
    today = date.today()

    days_until_tuesday = check_resting_days(1, today)
    days_until_thursday = check_resting_days(3, today)

    next_tuesday = convert_date(days_until_tuesday, today)
    next_thursday = convert_date(days_until_thursday, today)

    #add the hour (6 pm)
    next_tuesday+='-1800'
    next_thursday+='-1800'

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait)

    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printnig the results for the classes')
    #check the classes that are not available
    processed_classes, total_target_classes = check_classes(driver, all_buttons, class_names, next_tuesday, next_thursday)

    number_classes_user = summary(processed_classes, total_target_classes)

In [17]:
main()

Checking the classes...
Printnig the results for the classes
Already booked: Spin Class on Tomorrow (Tue, Nov 10)
Already booked: Spin Class on Thu, Nov 12

--- BOOKING SUMMARY ---
Classes booked:0
Waitlists joined:0
Already booked/waitlisted:2
Total Tuesday & Thursday 6pm classes: 0

--- DETAILED CLASS LIST ---
  • [Already Booked] Spin Class on Tomorrow (Tue, Nov 10)
  • [Already Booked] Spin Class on Thu, Nov 12


# Now is better but we have not finished, so next step **Verify Class bookings on the "My Bookings" Page**

* Navigate to the "My Bookings" page
* Count all Tuesday/Thursday 6pm bookings
* Compare the results with what you tried to book
* Print the verification results

> --- VERIFYING ON MY BOOKINGS PAGE ---
* ✓ Verified: Spin Class
* ✓ Verified: Spin Class (Waitlist)

> --- VERIFICATION RESULT ---
* Expected: 2 bookings
* Found: 2 bookings
* ✅ SUCCESS: All bookings verified!

If there is a mismatch, you can print out the difference between your verified bookings and your booking attempts:

> --- VERIFICATION RESULT ---
* Expected: 1 bookings
* Found: 2 bookings
* ❌ MISMATCH: Missing -1 bookings

# Modifying some classes

In [38]:
def log_in(wait, name, action, message=''):  
    #the functions take a tuple, [By.id, element] is the same
    locator = (By.ID, name)
    #if another button has an ID without button in it, it will not work, so we are going to change our logic
    if action=='click':  
        wait.until(
            #we get the element that we are looking
            EC.element_to_be_clickable((locator))  
        ).click()
    elif action=='type':
        input = wait.until(
            EC.visibility_of_element_located((locator))
        )
        input.send_keys(message)
    else:
        raise ValueError(f'Unknown action: {action}')

In [65]:
#this have an error since if no new classes are book for tuesday or thursday they will not be count
def check_classes(all_buttons, class_names, next_tuesday, next_thursday):
    processed_classes = []
    total_target_classes = 0
    #we return to all_buttons loop
    for button in all_buttons:
        #get the id
        button_id = button.get_attribute("id")

        # Check if this is Tuesday or Thursday at 6 PM
        is_target = check_day(
            button_id, next_tuesday, next_thursday
        )

        # Count every target class, regardless of its status
        # why because our instruction is for every class on that days have to be booked
        # so even if we have not booked that class we will, or if we have it will count them
        if is_target:
            total_target_classes += 1

        button_text = " ".join(button.text.split()).casefold()
        date_label = get_date_title(button)
        schedule_class = button_id.split("-")[2]
        class_name = class_names.get(schedule_class, schedule_class.title())

        status = None

        #all the rest is same 
        if button_text == "book class":
            if is_target:
                button.click()
                print(f"Clicked Booked Class for: {class_name} Class on {date_label}")
                status = "New Booking"
            
        elif button_text == "join waitlist":
            if is_target:                    
                button.click()
                print(f"Joined waitlist for: {class_name} Class on {date_label}")
                status = "New Waitlist"

        #check the rest   
        elif button_text == "booked":
            print(f"Already booked: {class_name} Class on {date_label}")
            status = "Already Booked"

        elif button_text == "waitlisted":
            print(f"Already on waitlist: {class_name} Class on {date_label}")
            status = "Already Waitlisted"

        #check for status
        if status is not None:
            processed_classes.append({
                "class": class_name,
                "date": date_label,
                "status": status
            })
        
    return processed_classes, total_target_classes 

In [68]:
def summary(processed_classes, total_target_classes):

    # Calculate counters
    new_bookings = sum(
        c["status"] == "New Booking"
        for c in processed_classes
    )

    new_waitlists = sum(
        c["status"] == "New Waitlist"
        for c in processed_classes
    )

    already_processed = sum(
        c["status"] in ("Already Booked", "Already Waitlisted")
        for c in processed_classes
    )
    #print
    print('\n--- BOOKING SUMMARY ---')
    print(f"Classes booked:{new_bookings}")
    print(f"Waitlists joined:{new_waitlists}") 
    print(f"Already booked/waitlisted:{already_processed}")
    #for the classes we print them:
    print(
        f"Total Tuesday & Thursday 6pm classes: {total_target_classes}"
    )

    print("\n--- DETAILED CLASS LIST ---")

    for class_info in processed_classes:

        print(
            f'  • [{class_info["status"]}] '
            f'{class_info["class"]} Class on {class_info["date"]}'
        )
    #return this value
    return new_bookings+new_waitlists+already_processed


In [63]:
def count_bookings(driver):
    #with find_elements in case we do not have any element we will have a [] empty
    booking_classes = driver.find_elements(By.CSS_SELECTOR, value="div[id^='booking-card-booking_']")
    waitlist_classes = driver.find_elements(By.CSS_SELECTOR, value="div[id^='waitlist-card-waitlist_']")
    print("\n--- VERIFYING ON MY BOOKINGS PAGE ---")

    #only the name   of the classes
    for booking in booking_classes:
        class_name = booking.find_element(By.CSS_SELECTOR, "h3").text.strip()
        print(f'✓ Verified: {class_name}')

    for booking in waitlist_classes:
        class_name = booking.find_element(By.CSS_SELECTOR, "h3").text.strip()
        print(f'✓ Verified: {class_name}')

    return len(booking_classes)+len(waitlist_classes)

In [64]:
def verifying_booking(number_booking, number_classes_user):
    #comapre 
    print('\n--- VERIFICATION RESULT ---')
    print(f'Expected:{number_classes_user} bookings')
    print(f'Found:{number_booking} bookings')

    if number_booking!=number_classes_user:
        print(f'❌ MISMATCH: Missing {number_booking-number_classes_user} bookings')
    else:
        print('✅ SUCCESS: All bookings verified!')

In [ ]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #calculate for tuesday and thursday
    today = date.today()

    days_until_tuesday = check_resting_days(1, today)
    days_until_thursday = check_resting_days(3, today)

    next_tuesday = convert_date(days_until_tuesday, today)

    next_thursday = convert_date(days_until_thursday, today)

    #add the hour (6 pm)
    next_tuesday+='-1800'
    next_thursday+='-1800'

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait, 'login-button', 'click')
    #enter email and password
    log_in(wait, 'email-input', 'type', ACCOUNT)
    log_in(wait, 'password-input', 'type', PASSWORD)
    #send them
    log_in(wait, 'submit-button', 'click')


    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printing the results for the classes')
    #check the classes that are not available
    processed_classes, total_target_classes = check_classes(all_buttons, class_names, next_tuesday, next_thursday)

    number_classes_user = summary(processed_classes, total_target_classes)
    #ACCESS TO BOOKING PAGE
    next_page = driver.find_element(By.LINK_TEXT, 'My Bookings')
    next_page.click() 
    #count bookings
    number_booking = count_bookings(driver)
    #compare
    verifying_booking(number_booking, number_classes_user)
    

In [66]:
main()

Checking the classes...
Printing the results for the classes
Already on waitlist: Spin Class on Today (Mon, Nov 9)
Already booked: Spin Class on Tomorrow (Tue, Nov 10)
Already booked: Spin Class on Thu, Nov 12

--- BOOKING SUMMARY ---
Classes booked:0
Waitlists joined:0
Already booked/waitlisted:3
Total Tuesday & Thursday 6pm classes: 2

--- DETAILED CLASS LIST ---
  • [Already Waitlisted] Spin Class on Today (Mon, Nov 9)
  • [Already Booked] Spin Class on Tomorrow (Tue, Nov 10)
  • [Already Booked] Spin Class on Thu, Nov 12

--- VERIFYING ON MY BOOKINGS PAGE ---
✓ Verified: Spin Class
✓ Verified: Spin Class
✓ Verified: Spin Class (Waitlist)

--- VERIFICATION RESULT ---
Expected:3 bookings
Found:3 bookings
✅ SUCCESS: All bookings verified!


# Now I put the date of the page 3 days from today, so lets see if we can have the right result

In [74]:
main()

2026-11-10
2026-11-10
Checking the classes...
Printing the results for the classes
2026-11-12-1700
2026-11-12-1800
Already booked: Spin Class on Today (Thu, Nov 12)
2026-11-12-1900
2026-11-13-0700
2026-11-13-0800
2026-11-13-0900
2026-11-13-1700
2026-11-13-1800
2026-11-13-1900
2026-11-14-0700
2026-11-14-0800
2026-11-14-0900
2026-11-14-1700
2026-11-14-1800
2026-11-14-1900
2026-11-15-0700
2026-11-15-0800
2026-11-15-0900
2026-11-15-1700
2026-11-15-1800
2026-11-15-1900
2026-11-16-0700
2026-11-16-0800
2026-11-16-0900
2026-11-16-1700
2026-11-16-1800
2026-11-16-1900
2026-11-17-0700
2026-11-17-0800
2026-11-17-0900
2026-11-17-1700
2026-11-17-1800
2026-11-17-1900
2026-11-18-0700
2026-11-18-0800
2026-11-18-0900
2026-11-18-1700
2026-11-18-1800
2026-11-18-1900

--- BOOKING SUMMARY ---
Classes booked:0
Waitlists joined:0
Already booked/waitlisted:1
Total Tuesday & Thursday 6pm classes: 1

--- DETAILED CLASS LIST ---
  • [Already Booked] Spin Class on Today (Thu, Nov 12)

--- VERIFYING ON MY BOOKINGS 

# I realized something important we were having the date with the library today, but that will not work if the page have a differente "today" so we can have the date from the title

In [98]:
from datetime import datetime

In [ ]:
def getToday(wait):
    title = wait.until(
        EC.visibility_of_element_located((By.CLASS_NAME, 'Schedule_scheduleTitle__zfZxg'))
    )
    title = title.text
    #Example of output, Class Schedule (Today: 11/9/2026)
    #we just wanted the date
    title = title.split('Today: ')[1].replace(")", "")
    #convert to datetime
    schedule_date = datetime.strptime(title, "%m/%d/%Y").date()
    return schedule_date
    

In [113]:
def main_temp():

    #Now we need to login first
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait, 'login-button', 'click')
    #enter email and password
    log_in(wait, 'email-input', 'type', ACCOUNT)
    log_in(wait, 'password-input', 'type', PASSWORD)
    #send them
    log_in(wait, 'submit-button', 'click')

    #here we get the date from the page

    #calculate for tuesday and thursday
    today = getToday(wait)

    days_until_tuesday = check_resting_days(1, today)
    days_until_thursday = check_resting_days(3, today)

    next_tuesday = convert_date(days_until_tuesday, today)
    print(next_tuesday)
    next_thursday = convert_date(days_until_thursday, today)
    print(next_thursday)
    #add the hour (6 pm)
    next_tuesday+='-1800'
    next_thursday+='-1800'

    print(next_tuesday)
    print(next_thursday)

    

In [115]:
main_temp()

2026-11-17
2026-11-12
2026-11-17-1800
2026-11-12-1800


# Now we have all, lets see it complete

In [116]:
#libraries
from pathlib import Path
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
#we have to wait until the page is charge so 
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException
from datetime import datetime, timedelta 

In [117]:
env_file = Path.cwd().parent / '.env'
load_dotenv(env_file)
PASSWORD = os.getenv('GYM_PASSWORD')
ACCOUNT = os.getenv('GYM_ACCOUNT')
GYM_URL = "https://appbrewery.github.io/gym/"

In [118]:
def chrome_configuration():
    #stay open using the Chrome option:
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_experimental_option("detach", True)
    #Create a folder for the Chrome Profile selenium AND will use it every time
    user_data_dir = os.path.join(os.getcwd(), "chrome_profile")
    #use the directory specified to store a "profile".
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")
    return chrome_options


In [119]:
def make_connection(chrome_options):
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(GYM_URL)
    return driver

In [120]:
def log_in(wait, name, action, message=''):  
    #the functions take a tuple, [By.id, element] is the same
    locator = (By.ID, name)
    #if another button has an ID without button in it, it will not work, so we are going to change our logic
    if action=='click':  
        wait.until(
            #we get the element that we are looking
            EC.element_to_be_clickable((locator))  
        ).click()
    elif action=='type':
        input = wait.until(
            EC.visibility_of_element_located((locator))
        )
        input.send_keys(message)
    else:
        raise ValueError(f'Unknown action: {action}')

In [122]:
def check_resting_days(amount,today):
    days_until = (amount - today.weekday()) % 7
    '''
    if days_until == 0:
        days_until = 7
    '''
    return days_until

In [123]:
def convert_date(days_until, today):
    next_date = today + timedelta(days=days_until)
    #and we convert the date into a string
    date_string = next_date.strftime("%Y-%m-%d")
    return date_string

In [124]:
def check_day(button, next_tuesday, next_thursday):
    #all at once
    date_tag = ('-').join((button.split('-')[3:]))
    #and then we compare
    #We can return the Boolean expression directly. There's no need for an additional if.
    return date_tag in (next_tuesday, next_thursday)

In [125]:
def getToday(wait):
    title = wait.until(
        EC.visibility_of_element_located((By.CLASS_NAME, 'Schedule_scheduleTitle__zfZxg'))
    )
    title = title.text
    #Example of output, Class Schedule (Today: 11/9/2026)
    #we just wanted the date
    title = title.split('Today: ')[1].replace(")", "")
    #convert to datetime
    schedule_date = datetime.strptime(title, "%m/%d/%Y").date()
    return schedule_date

In [126]:
def tuesday_thursday(wait):
    #calculate for tuesday and thursday
    today = getToday(wait)

    days_until_tuesday = check_resting_days(1, today)
    days_until_thursday = check_resting_days(3, today)

    next_tuesday = convert_date(days_until_tuesday, today)
    next_thursday = convert_date(days_until_thursday, today)
    #add the hour (6 pm)
    next_tuesday+='-1800'
    next_thursday+='-1800'

    return next_tuesday, next_thursday

In [121]:
def find_all_buttons(wait):
    #we will wait until the elements (all the buttons) are displayed (loaded)
    list_buttons = wait.until(
        EC.visibility_of_all_elements_located((
        By.CSS_SELECTOR, 
        "button[id^='book-button-']"
        ))
    )
    return list_buttons

In [127]:
def get_date_title(button):
    # Get date from the ancestor
    day_group = button.find_element(
        By.XPATH,
        "ancestor::div[starts-with(@id, 'day-group-')][1]"
    )

    day_title = day_group.find_element(
        By.CSS_SELECTOR, "h2"
    ).text.strip()
    
    return day_title

In [128]:
def check_classes(all_buttons, class_names, next_tuesday, next_thursday):
    processed_classes = []
    total_target_classes = 0

    for button in all_buttons:
        #get the id
        button_id = button.get_attribute("id")

        is_target = check_day(
            button_id, next_tuesday, next_thursday
        )

        if is_target:
            total_target_classes += 1

        button_text = " ".join(button.text.split()).casefold()
        date_label = get_date_title(button)
        schedule_class = button_id.split("-")[2]
        class_name = class_names.get(schedule_class, schedule_class.title())

        status = None

        #all the rest is same 
        if button_text == "book class":
            if is_target:
                button.click()
                print(f"Clicked Booked Class for: {class_name} Class on {date_label}")
                status = "New Booking"
            
        elif button_text == "join waitlist":
            if is_target:                    
                button.click()
                print(f"Joined waitlist for: {class_name} Class on {date_label}")
                status = "New Waitlist"

        #check the rest   
        elif button_text == "booked":
            print(f"Already booked: {class_name} Class on {date_label}")
            status = "Already Booked"

        elif button_text == "waitlisted":
            print(f"Already on waitlist: {class_name} Class on {date_label}")
            status = "Already Waitlisted"

        #check for status
        if status is not None:
            processed_classes.append({
                "class": class_name,
                "date": date_label,
                "status": status
            })
        
    return processed_classes, total_target_classes 

In [129]:
def summary(processed_classes, total_target_classes):

    # Calculate counters
    new_bookings = sum(
        c["status"] == "New Booking"
        for c in processed_classes
    )

    new_waitlists = sum(
        c["status"] == "New Waitlist"
        for c in processed_classes
    )

    already_processed = sum(
        c["status"] in ("Already Booked", "Already Waitlisted")
        for c in processed_classes
    )
    #print
    print('\n--- BOOKING SUMMARY ---')
    print(f"Classes booked:{new_bookings}")
    print(f"Waitlists joined:{new_waitlists}") 
    print(f"Already booked/waitlisted:{already_processed}")
    #for the classes we print them:
    print(
        f"Total Tuesday & Thursday 6pm classes: {total_target_classes}"
    )

    print("\n--- DETAILED CLASS LIST ---")

    for class_info in processed_classes:

        print(
            f'  • [{class_info["status"]}] '
            f'{class_info["class"]} Class on {class_info["date"]}'
        )
    #return this value
    return new_bookings+new_waitlists+already_processed

In [130]:
def count_bookings(driver):
    #with find_elements in case we do not have any element we will have a [] empty
    booking_classes = driver.find_elements(By.CSS_SELECTOR, value="div[id^='booking-card-booking_']")
    waitlist_classes = driver.find_elements(By.CSS_SELECTOR, value="div[id^='waitlist-card-waitlist_']")
    print("\n--- VERIFYING ON MY BOOKINGS PAGE ---")

    #only the name   of the classes
    for booking in booking_classes:
        class_name = booking.find_element(By.CSS_SELECTOR, "h3").text.strip()
        print(f'✓ Verified: {class_name}')

    for booking in waitlist_classes:
        class_name = booking.find_element(By.CSS_SELECTOR, "h3").text.strip()
        print(f'✓ Verified: {class_name}')

    return len(booking_classes)+len(waitlist_classes)
def verifying_booking(number_booking, number_classes_user):
    #comapre 
    print('\n--- VERIFICATION RESULT ---')
    print(f'Expected:{number_classes_user} bookings')
    print(f'Found:{number_booking} bookings')

    if number_booking!=number_classes_user:
        print(f'❌ MISMATCH: Missing {number_booking-number_classes_user} bookings')
    else:
        print('✅ SUCCESS: All bookings verified!')

In [133]:
def main():
    #make a dictionary to print the classes
    class_names = {
        "yoga": "Yoga",
        "spin": "Spin",
        "hiit": "HIIT"
    }

    #initialize chrome page with the options
    chrome_options = chrome_configuration()
    driver = make_connection(chrome_options)

    #make a waiter 
    wait = WebDriverWait(
        driver,
        15, #wait 15 seconds
        poll_frequency=0.2, #wait 0.2 seconds
        ignored_exceptions=(StaleElementReferenceException) # we ignore this exception
    )

    #log in
    log_in(wait, 'login-button', 'click')
    #enter email and password
    log_in(wait, 'email-input', 'type', ACCOUNT)
    log_in(wait, 'password-input', 'type', PASSWORD)
    #send them
    log_in(wait, 'submit-button', 'click')

    next_tuesday, next_thursday = tuesday_thursday(wait)

    #the find_all_buttons
    print('Checking the classes...')
    all_buttons = find_all_buttons(wait)

    print('Printing the results for the classes')
    #check the classes that are not available
    processed_classes, total_target_classes = check_classes(all_buttons, class_names, next_tuesday, next_thursday)

    number_classes_user = summary(processed_classes, total_target_classes)
    #ACCESS TO BOOKING PAGE
    next_page = driver.find_element(By.LINK_TEXT, 'My Bookings')
    next_page.click() 
    #count bookings
    number_booking = count_bookings(driver)
    #compare
    verifying_booking(number_booking, number_classes_user)

In [134]:
main()

Checking the classes...
Printing the results for the classes
Clicked Booked Class for: Spin Class on Today (Thu, Nov 12)
Joined waitlist for: Spin Class on Tue, Nov 17

--- BOOKING SUMMARY ---
Classes booked:1
Waitlists joined:1
Already booked/waitlisted:0
Total Tuesday & Thursday 6pm classes: 2

--- DETAILED CLASS LIST ---
  • [New Booking] Spin Class on Today (Thu, Nov 12)
  • [New Waitlist] Spin Class on Tue, Nov 17

--- VERIFYING ON MY BOOKINGS PAGE ---
✓ Verified: Spin Class
✓ Verified: Spin Class (Waitlist)

--- VERIFICATION RESULT ---
Expected:2 bookings
Found:2 bookings
✅ SUCCESS: All bookings verified!


# WE will do it on separeted files